# 02 · Data Cleaning & Transformation — Agentris
Soto Batok & Penyetan Pak Tris | A26-CS193

Notebook lengkap untuk membersihkan empat CSV transaksi dengan referensi 23 menu. Penjualan tetap **satu CSV** (satu baris per detail pesanan). Data primer dan sintetis dipisahkan. File raw tidak ditimpa.

Kebijakan: nominal sintetis menggunakan Rupiah, harga referensi dianggap tetap selama periode simulasi (Agustus–Oktober 2026), tanpa diskon atau ongkos tambahan. Koreksi berbasis aturan hanya berlaku pada simulasi; nilai asli tersimpan dalam raw dan log. Jangan menebak menu, tanggal atau nominal yang tidak dapat dipulihkan. Pengeluaran dengan keterangan hilang tetapi tanggal dan nominal valid tetap dipakai untuk rekap dengan label `Tidak tercatat`.

In [14]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Bisa dijalankan dari root proyek maupun folder notebooks.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'datasets' / 'raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Buka notebook dari folder proyek Agentris yang sudah diekstrak.')
RAW = ROOT / 'datasets' / 'raw'
REF = ROOT / 'datasets' / 'reference'
INTERIM = ROOT / 'datasets' / 'interim'
PROCESSED = ROOT / 'datasets' / 'processed'
REPORTS = ROOT / 'reports'
for folder in [INTERIM, PROCESSED, REPORTS / 'figures']:
    folder.mkdir(parents=True, exist_ok=True)
FILES = {
    'penjualan_sintetis': RAW / 'transaksi_penjualan_sintetis_raw_gabungan.csv',
    'pengeluaran_sintetis': RAW / 'transaksi_pengeluaran_sintetis_raw.csv',
    'penjualan_primer': RAW / 'transaksi_penjualan_primer_raw.csv',
    'pengeluaran_primer': RAW / 'transaksi_pengeluaran_primer_raw.csv',
    'menu': REF / 'referensi_menu.csv',
}
def load_raw():
    return {key: pd.read_csv(path, dtype='string') for key, path in FILES.items()}
print('Project root:', ROOT)

Project root: d:\1.KULIAH KUYY\Studi Independent\agentris\data-science


In [15]:
import re
from datetime import datetime
from decimal import Decimal, InvalidOperation

data = load_raw()
raw_j = data['penjualan_sintetis']
raw_b = data['pengeluaran_sintetis']
menu = data['menu'].copy()
changes, rejects = [], []

def log(dataset, key, col, before, after, reason):
    changes.append(dict(dataset=dataset, id=key, kolom=col,
                        sebelum=before, sesudah=after, alasan=reason))

def reject(dataset, key, reason):
    rejects.append(dict(dataset=dataset, id=key, alasan=reason))

def assessment(df):
    return pd.DataFrame({'tipe':df.dtypes.astype(str), 'missing':df.isna().sum(),
                         'unik':df.nunique()})
for name, df in data.items():
    print(name, df.shape, '| duplikat identik:', df.duplicated().sum())
    display(assessment(df))

penjualan_sintetis (2694, 9) | duplikat identik: 43


,tipe,missing,unik
id_transaksi,string,0,1183
tanggal,string,8,132
nama_pembeli,string,6,190
jenis_makanan,string,10,51
jumlah,string,15,5
harga_satuan,string,10,21
subtotal,string,5,31
id_detail,string,5,2646
total_transaksi,string,3,135


pengeluaran_sintetis (471, 4) | duplikat identik: 7


,tipe,missing,unik
no,string,0,464
tanggal,string,1,109
keterangan,string,4,24
total,string,1,140


penjualan_primer (11, 5) | duplikat identik: 0


,tipe,missing,unik
no,string,0,11
tanggal,string,0,1
nama_pembeli,string,0,10
jenis_makanan,string,11,0
total,string,0,8


pengeluaran_primer (13, 4) | duplikat identik: 0


,tipe,missing,unik
no,string,0,13
tanggal,string,13,0
keterangan,string,0,10
total,string,0,11


menu (23, 4) | duplikat identik: 0


,tipe,missing,unik
id_menu,string,0,23
kategori,string,0,5
nama_menu,string,0,23
harga,string,0,12


## 1. Menghapus duplikasi identik
ID transaksi dapat berulang karena pesanan terdiri atas beberapa item. Hanya salinan baris yang identik dihapus. Konflik ID detail atau ID pengeluaran dikarantina, bukan dipilih sembarang. Catatan primer yang tampak berulang dipertahankan.

In [16]:
def dedup(df, dataset, key):
    mask = df.duplicated(keep='first')
    for _, r in df[mask].iterrows():
        log(dataset, r[key], 'baris', r.to_json(force_ascii=False), None,
            'Duplikat identik; satu salinan dipertahankan')
    return df.loc[~mask].copy()

jual = dedup(raw_j, 'penjualan_sintetis', 'id_transaksi')
belanja = dedup(raw_b, 'pengeluaran_sintetis', 'no')
print('Baris duplikat dihapus:', len(raw_j)-len(jual), len(raw_b)-len(belanja))

Baris duplikat dihapus: 43 7


## 2. Fungsi normalisasi
Format tanggal diproses eksplisit (hari/bulan/tahun untuk tanggal lokal). Tanggal tidak sah menjadi kosong. Nominal seperti `Rp 13.000`, `37.5 rb`, `28.000` dikonversi ke integer Rupiah. Skala ribuan khusus catatan primer belanja diproses terpisah.

In [17]:
def tidy(x):
    return None if pd.isna(x) or not str(x).strip() else ' '.join(str(x).split())

def norm_date(x):
    s = tidy(x)
    if s is None: return None
    months = {'jan':'01','feb':'02','mar':'03','apr':'04','mei':'05','jun':'06',
              'jul':'07','agu':'08','sep':'09','okt':'10','nov':'11','des':'12'}
    m = re.fullmatch(r'(\d{1,2})\s+([a-zA-Z]{3})\s+(\d{4})', s)
    if m and m[2].lower() in months:
        s = f'{m[1]}/{months[m[2].lower()]}/{m[3]}'
    for fmt in ('%Y-%m-%d','%Y/%m/%d','%d/%m/%Y','%d-%m-%Y'):
        try: return datetime.strptime(s,fmt).date().isoformat()
        except ValueError: pass
    return None

def money(x, ribuan=False):
    s = tidy(x)
    if s is None: return None
    s = re.sub(r'^rp\s*','',s.lower()).replace(' ','')
    scale = 1000 if ribuan else 1
    if s.endswith('rb') or s.endswith('ribu'):
        s = re.sub(r'(ribu|rb)$','',s)
        scale = 1000
    if scale == 1000:
        if not re.fullmatch(r'-?\d+(?:[.,]\d+)?',s): return None
        s = s.replace(',','.')
    elif re.fullmatch(r'-?\d{1,3}(?:\.\d{3})+',s):
        s = s.replace('.','')
    elif not re.fullmatch(r'-?\d+',s): return None
    try:
        v = Decimal(s)*scale
        return int(v) if v == v.to_integral_value() else None
    except (InvalidOperation, ValueError): return None

def qty(x):
    s = tidy(x)
    return int(s) if s is not None and re.fullmatch(r'[1-9]\d*',s) else None

def in_period(d):
    return d is not None and '2026-08-01' <= d <= '2026-10-31'

assert money('Rp 13.000') == 13000 and money('37.5 rb') == 37500
assert money('24,5',ribuan=True) == 24500 and money('28') == 28
assert norm_date('31/09/2026') is None and norm_date('31 Agu 2026') == '2026-08-31'
assert norm_date('04/09/2026') == '2026-09-04'
print('Uji fungsi normalisasi: lulus')

Uji fungsi normalisasi: lulus


## 3. Standardisasi nama menu dan pembeli
Nama menu disamakan dengan daftar primer melalui trim, casefold dan koreksi typo `sotto` → `soto`. Tanda `+` pada nama menu resmi tetap dipertahankan. Nama menu yang kosong tidak ditebak berdasarkan harga. Nama pembeli kosong dipulihkan dari detail dalam ID yang sama, atau diberi label anonim jika seluruh detail tidak memiliki nama.

In [18]:
price = dict(zip(menu.nama_menu, menu.harga.map(money)))
menu_lookup = {tidy(s).casefold():s for s in menu.nama_menu}
assert len(price) == 23 and all(v is not None and v > 0 for v in price.values())

def norm_menu(x):
    s = tidy(x)
    if s is None: return None
    key = re.sub(r'^sotto','soto',s.casefold())
    return menu_lookup.get(key)

def norm_buyer(x):
    s = tidy(x)
    if s is None: return None
    k = s.casefold()
    if k in ('shopeefood','gofood','pembeli tanpa nama'):
        return {'shopeefood':'ShopeeFood','gofood':'GoFood','pembeli tanpa nama':'Pembeli tanpa nama'}[k]
    return s.title()

def unique_valid(values):
    return set(v for v in values if v is not None)

def record_changes(dataset, key, old, new, reasons):
    for c, after in new.items():
        before = old[c]
        if (None if pd.isna(before) else str(before)) != (None if after is None else str(after)):
            log(dataset,key,c,before,after,reasons.get(c,'Normalisasi format'))

display(menu[['nama_menu','harga']].head())

,nama_menu,harga
0,Soto ayam,10000
1,Soto ayam pisah,13000
2,Soto sapi,14000
3,Soto sapi pisah,16000
4,Soto ayam bungkus + nasi,12000


## 4. Cleaning penjualan per ID transaksi
Tanggal/pembeli dipulihkan hanya dari detail yang konsisten. Kuantitas kosong/nol dipulihkan jika subtotal merupakan kelipatan harga referensi dan harga tercatat tidak bertentangan. Harga satuan mengikuti daftar menu, subtotal = jumlah × harga, dan total = jumlah seluruh subtotal. Semua perubahan dicatat. Jika satu detail tak dapat dipulihkan, seluruh transaksi dikarantina agar tidak menghitung pendapatan parsial. Rekonstruksi total hanya dilakukan untuk data sintetis dengan rincian lengkap.

In [19]:
clean_rows = []
conflict_ids = set(jual.loc[jual.id_detail.notna() & jual.id_detail.duplicated(keep=False),'id_transaksi'])
for key, g in jual.groupby('id_transaksi',sort=False,dropna=False):
    reasons = []
    if pd.isna(key): reasons.append('ID transaksi kosong')
    if key in conflict_ids: reasons.append('ID detail konflik')
    dates = unique_valid(norm_date(x) for x in g.tanggal)
    buyers = unique_valid(norm_buyer(x) for x in g.nama_pembeli)
    if len(dates) != 1 or not all(in_period(d) for d in dates): reasons.append('Tanggal tidak dapat dipulihkan atau konflik')
    if len(buyers) > 1: reasons.append('Nama pembeli konflik')
    date = next(iter(dates)) if len(dates)==1 else None
    buyer = next(iter(buyers)) if len(buyers)==1 else 'Pembeli tanpa nama'
    pending = []
    for _, r in g.iterrows():
        item = norm_menu(r.jenis_makanan)
        detail = tidy(r.id_detail)
        if detail is None or item is None:
            reasons.append('Rincian ID/menu hilang atau menu tidak dikenali'); continue
        p = price[item]; q = qty(r.jumlah)
        s = money(r.subtotal); recorded_price = money(r.harga_satuan)
        if q is None:
            if s is not None and s > 0 and s % p == 0 and recorded_price in (None,p): q = s//p
            else:
                reasons.append('Kuantitas tidak dapat dipulihkan'); continue
        row = dict(id_transaksi=key,tanggal=date,nama_pembeli=buyer,jenis_makanan=item,
                   jumlah=q,harga_satuan=p,subtotal=q*p,id_detail=detail,total_transaksi=None)
        pending.append((r,row))
    if reasons:
        reject('penjualan_sintetis',key,'; '.join(dict.fromkeys(reasons)))
        continue
    grand = sum(row['subtotal'] for _,row in pending)
    # Satu transaksi harus memiliki setidaknya satu rincian lengkap.
    if not pending or grand <= 0:
        reject('penjualan_sintetis',key,'Tidak ada rincian valid'); continue
    for r,row in pending:
        row['total_transaksi'] = grand
        record_changes('penjualan_sintetis',key,r,row,{
            'tanggal':'Format ISO atau tanggal konsisten dari ID yang sama',
            'nama_pembeli':'Standardisasi atau label anonim',
            'jenis_makanan':'Nama baku dari referensi menu',
            'jumlah':'Pulihkan dari subtotal / harga referensi jika jumlah tidak valid',
            'harga_satuan':'Harga referensi menu untuk simulasi',
            'subtotal':'Hitung ulang jumlah × harga referensi',
            'total_transaksi':'Hitung ulang seluruh subtotal dalam ID transaksi'})
        clean_rows.append(row)
jual_clean = pd.DataFrame(clean_rows,columns=raw_j.columns)
print('Rincian penjualan valid:',len(jual_clean),'| transaksi unik:',jual_clean.id_transaksi.nunique())
display(jual_clean.head(10))

Rincian penjualan valid: 2573 | transaksi unik: 1148


,id_transaksi,tanggal,nama_pembeli,jenis_makanan,jumlah,harga_satuan,subtotal,id_detail,total_transaksi
0,PJ-00001,2026-08-01,ShopeeFood,Soto ayam pisah,1,13000,13000,PJ-00001-01,13000
1,PJ-00002,2026-08-01,Pelanggan 179,Pecel lele,1,13000,13000,PJ-00002-01,16000
2,PJ-00002,2026-08-01,Pelanggan 179,Teh es,1,3000,3000,PJ-00002-02,16000
3,PJ-00003,2026-08-01,Pelanggan 146,Soto ayam,1,10000,10000,PJ-00003-01,10000
4,PJ-00004,2026-08-01,ShopeeFood,Soto ayam bungkus + nasi,1,12000,12000,PJ-00004-01,15000
5,PJ-00004,2026-08-01,ShopeeFood,Teh panas,1,3000,3000,PJ-00004-02,15000
6,PJ-00005,2026-08-01,GoFood,Pecel lele,1,13000,13000,PJ-00005-01,19000
7,PJ-00005,2026-08-01,GoFood,Kacang halus,1,6000,6000,PJ-00005-02,19000
8,PJ-00006,2026-08-01,Pelanggan 088,Nasi goreng teri hijau,1,20000,20000,PJ-00006-01,23000
9,PJ-00006,2026-08-01,Pelanggan 088,Teh es,1,3000,3000,PJ-00006-02,23000


## 5. Cleaning pengeluaran sintetis
Normalisasi format tanggal, nominal, dan keterangan. Keterangan yang hilang diberi `Tidak tercatat` agar nilai belanja yang valid tidak hilang. Nominal nol, negatif, kosong, atau tanggal yang tidak diketahui dikarantina; tidak diubah menjadi nilai positif tanpa bukti.

In [20]:
expense_lookup = {s.casefold():s for s in ['Beras','Telur','Bumbu racik','Tempe','Minyak goreng','Bahan donat','Jeruk','Parkir','Sapi','Sayuran','Nila','Ayam','Lele','Kemasan']}
expense_rows = []
conflicts = set(belanja.loc[belanja.no.duplicated(keep=False),'no'])
for _,r in belanja.iterrows():
    key = r.no; d = norm_date(r.tanggal); amount = money(r.total); desc = tidy(r.keterangan)
    issues = []
    if pd.isna(key) or key in conflicts: issues.append('ID pengeluaran kosong atau konflik')
    if not in_period(d): issues.append('Tanggal kosong, tidak sah, atau di luar periode')
    if amount is None or amount <= 0: issues.append('Nominal kosong atau tidak positif')
    if issues:
        reject('pengeluaran_sintetis',key,'; '.join(issues)); continue
    desc = expense_lookup.get(desc.casefold(),desc.capitalize()) if desc else 'Tidak tercatat'
    row = dict(no=key,tanggal=d,keterangan=desc,total=amount)
    record_changes('pengeluaran_sintetis',key,r,row,{'keterangan':'Standardisasi atau label Tidak tercatat','total':'Konversi integer Rupiah'})
    expense_rows.append(row)
belanja_clean = pd.DataFrame(expense_rows,columns=raw_b.columns)
print('Pengeluaran valid:',len(belanja_clean))
display(belanja_clean.head(10))

Pengeluaran valid: 457


,no,tanggal,keterangan,total
0,PG-00001,2026-08-01,Bumbu racik,11500
1,PG-00002,2026-08-01,Beras,17000
2,PG-00003,2026-08-01,Tempe,11000
3,PG-00004,2026-08-01,Nila,40500
4,PG-00005,2026-08-02,Minyak goreng,19000
5,PG-00006,2026-08-02,Nila,21000
6,PG-00008,2026-08-02,Ayam,51000
7,PG-00009,2026-08-02,Tempe,11000
8,PG-00010,2026-08-02,Telur,29000
9,PG-00011,2026-08-02,Beras,34000


## 6. Cleaning primer dan rekonsiliasi sumber
Catatan primer penjualan berisi hari dan tanggal tanpa tahun, serta tidak memuat rincian menu. Catatan primer pengeluaran tidak memiliki tanggal. Angka pengeluaran dibaca sebagai ribuan Rupiah sesuai interpretasi sumber; masih perlu konfirmasi pemilik. Simpan sebagai berkas review tersendiri, tidak digabung ke rekap sintetis. Ejaan `Jeeuk` diduga `Jeruk`; `Cadar` belum dapat dipastikan. Entri berulang tidak dihapus.

In [21]:
primer_jual = data['penjualan_primer'].copy()
primer_belanja = data['pengeluaran_primer'].copy()
for df, dataset, scale in [(primer_jual,'penjualan_primer',False),(primer_belanja,'pengeluaran_primer',True)]:
    vals = df.total.map(lambda x: money(x,ribuan=scale))
    for idx, val in vals.items():
        log(dataset,df.at[idx,'no'],'total',df.at[idx,'total'],val,'Normalisasi Rupiah; skala ribuan asumsi sumber' if scale else 'Normalisasi Rupiah')
    df['total'] = vals.astype('Int64')
    df['tanggal_asli'] = df['tanggal']
    df['tanggal'] = pd.NA  # tanggal ISO belum diketahui
    df['status_review'] = 'Perlu konfirmasi tanggal lengkap dan rincian menu' if not scale else 'Perlu konfirmasi tanggal dan satuan ribuan'
for idx in primer_jual.index[primer_jual.nama_pembeli.str.casefold().isin(['shoopee','shopee'])]:
    log('penjualan_primer',primer_jual.at[idx,'no'],'nama_pembeli',primer_jual.at[idx,'nama_pembeli'],'Shopee','Standardisasi ejaan kanal; jenis layanan tidak dipastikan')
    primer_jual.at[idx,'nama_pembeli']='Shopee'
for idx in primer_belanja.index[primer_belanja.keterangan.eq('Jeeuk')]:
    primer_belanja.at[idx,'status_review'] += '; dugaan Jeruk, konfirmasi ejaan'
for idx in primer_belanja.index[primer_belanja.keterangan.eq('Cadar')]:
    primer_belanja.at[idx,'status_review'] += '; konfirmasi nama Cadar'
assert primer_jual.total.sum()==285000 and primer_belanja.total.sum()==269000
print('Rekonsiliasi nominal primer (interpretasi sumber):',primer_jual.total.sum(),primer_belanja.total.sum())
display(primer_jual)
display(primer_belanja)

Rekonsiliasi nominal primer (interpretasi sumber): 285000 269000


,no,tanggal,nama_pembeli,jenis_makanan,total,tanggal_asli,status_review
0,AS-001,<NA>,Shopee,<NA>,10000,Senin 07 sept,Perlu konfirmasi tanggal lengkap dan rincian menu
1,AS-002,<NA>,Pembeli P01,<NA>,11000,Senin 07 sept,Perlu konfirmasi tanggal lengkap dan rincian menu
2,AS-003,<NA>,Pembeli P02,<NA>,11000,Senin 07 sept,Perlu konfirmasi tanggal lengkap dan rincian menu
3,AS-004,<NA>,Stpn,<NA>,40000,Senin 07 sept,Perlu konfirmasi tanggal lengkap dan rincian menu
4,AS-005,<NA>,Pembeli P03,<NA>,80000,Senin 07 sept,Perlu konfirmasi tanggal lengkap dan rincian menu
5,AS-006,<NA>,Pembeli P04,<NA>,25000,Senin 07 sept,Perlu konfirmasi tanggal lengkap dan rincian menu
6,AS-007,<NA>,Ojek,<NA>,17000,Senin 07 sept,Perlu konfirmasi tanggal lengkap dan rincian menu
7,AS-008,<NA>,Ojek,<NA>,17000,Senin 07 sept,Perlu konfirmasi tanggal lengkap dan rincian menu
8,AS-009,<NA>,Pembeli P05,<NA>,17000,Senin 07 sept,Perlu konfirmasi tanggal lengkap dan rincian menu
9,AS-010,<NA>,Pembeli P06,<NA>,42000,Senin 07 sept,Perlu konfirmasi tanggal lengkap dan rincian menu


,no,tanggal,keterangan,total,tanggal_asli,status_review
0,AG-001,<NA>,Lele,28000,<NA>,Perlu konfirmasi tanggal dan satuan ribuan
1,AG-002,<NA>,Nila,20000,<NA>,Perlu konfirmasi tanggal dan satuan ribuan
2,AG-003,<NA>,Ayam,46000,<NA>,Perlu konfirmasi tanggal dan satuan ribuan
3,AG-004,<NA>,Sapi,35000,<NA>,Perlu konfirmasi tanggal dan satuan ribuan
4,AG-005,<NA>,Cadar,32000,<NA>,Perlu konfirmasi tanggal dan satuan ribuan; ko...
5,AG-006,<NA>,Jeeuk,10000,<NA>,Perlu konfirmasi tanggal dan satuan ribuan; du...
6,AG-007,<NA>,Puyuh,13500,<NA>,Perlu konfirmasi tanggal dan satuan ribuan
7,AG-008,<NA>,Racik,4000,<NA>,Perlu konfirmasi tanggal dan satuan ribuan
8,AG-009,<NA>,Parkir,1000,<NA>,Perlu konfirmasi tanggal dan satuan ribuan
9,AG-010,<NA>,Tempe,7000,<NA>,Perlu konfirmasi tanggal dan satuan ribuan


## 7. Pemeriksaan kualitas setelah cleaning
Periksa jumlah baris, ID, kelengkapan, harga, dan konsistensi total. Bandingkan cakupan raw dengan hasil cleaning. Total transaksi tidak dijumlahkan berulang; omzet berasal dari subtotal. Log karantina bukan bukti transaksi tidak terjadi.

In [22]:
reject_log = pd.DataFrame(rejects,columns=['dataset','id','alasan'])
change_log = pd.DataFrame(changes,columns=['dataset','id','kolom','sebelum','sesudah','alasan'])
summary = pd.DataFrame([
    {'dataset':'Penjualan sintetis','baris_raw':len(raw_j),'duplikat_dihapus':len(raw_j)-len(jual),'baris_karantina':len(jual)-len(jual_clean),'baris_clean':len(jual_clean),'id_raw':raw_j.id_transaksi.nunique(),'id_clean':jual_clean.id_transaksi.nunique()},
    {'dataset':'Pengeluaran sintetis','baris_raw':len(raw_b),'duplikat_dihapus':len(raw_b)-len(belanja),'baris_karantina':len(belanja)-len(belanja_clean),'baris_clean':len(belanja_clean),'id_raw':raw_b.no.nunique(),'id_clean':belanja_clean.no.nunique()}
])
display(summary)
display(reject_log)
display(change_log.head(20))
assert not jual_clean.isna().any().any() and not belanja_clean.isna().any().any()
assert jual_clean.id_detail.is_unique and belanja_clean.no.is_unique
assert jual_clean.harga_satuan.eq(jual_clean.jenis_makanan.map(price)).all()
assert jual_clean.subtotal.eq(jual_clean.jumlah*jual_clean.harga_satuan).all()
group = jual_clean.groupby('id_transaksi')
assert group.total_transaksi.nunique().eq(1).all()
assert group.subtotal.sum().eq(group.total_transaksi.first()).all()
assert group.tanggal.nunique().eq(1).all() and group.nama_pembeli.nunique().eq(1).all()
for name, raw, clean, key in [('penjualan_sintetis',raw_j,jual_clean,'id_transaksi'),('pengeluaran_sintetis',raw_b,belanja_clean,'no')]:
    assert set(raw[key])-set(clean[key]) == set(reject_log.loc[reject_log.dataset.eq(name),'id'])
print('Pemeriksaan awal lulus. Omzet sintetis kandidat:',jual_clean.subtotal.sum())
print('Pengeluaran sintetis kandidat:',belanja_clean.total.sum())

,dataset,baris_raw,duplikat_dihapus,baris_karantina,baris_clean,id_raw,id_clean
0,Penjualan sintetis,2694,43,78,2573,1183,1148
1,Pengeluaran sintetis,471,7,7,457,464,457


,dataset,id,alasan
0,penjualan_sintetis,PJ-00044,Rincian ID/menu hilang atau menu tidak dikenali
1,penjualan_sintetis,PJ-00048,Rincian ID/menu hilang atau menu tidak dikenali
2,penjualan_sintetis,PJ-00091,Rincian ID/menu hilang atau menu tidak dikenali
3,penjualan_sintetis,PJ-00148,Tanggal tidak dapat dipulihkan atau konflik
4,penjualan_sintetis,PJ-00176,Tanggal tidak dapat dipulihkan atau konflik
5,penjualan_sintetis,PJ-00192,Tanggal tidak dapat dipulihkan atau konflik
6,penjualan_sintetis,PJ-00203,Rincian ID/menu hilang atau menu tidak dikenali
7,penjualan_sintetis,PJ-00214,Rincian ID/menu hilang atau menu tidak dikenali
8,penjualan_sintetis,PJ-00312,Rincian ID/menu hilang atau menu tidak dikenali
9,penjualan_sintetis,PJ-00337,Rincian ID/menu hilang atau menu tidak dikenali


,dataset,id,kolom,sebelum,sesudah,alasan
0,penjualan_sintetis,PJ-00119,baris,"{""id_transaksi"":""PJ-00119"",""tanggal"":""2026-08-...",None,Duplikat identik; satu salinan dipertahankan
1,penjualan_sintetis,PJ-00119,baris,"{""id_transaksi"":""PJ-00119"",""tanggal"":""2026-08-...",None,Duplikat identik; satu salinan dipertahankan
2,penjualan_sintetis,PJ-00119,baris,"{""id_transaksi"":""PJ-00119"",""tanggal"":""2026-08-...",None,Duplikat identik; satu salinan dipertahankan
3,penjualan_sintetis,PJ-00221,baris,"{""id_transaksi"":""PJ-00221"",""tanggal"":""2026-08-...",None,Duplikat identik; satu salinan dipertahankan
4,penjualan_sintetis,PJ-00221,baris,"{""id_transaksi"":""PJ-00221"",""tanggal"":""2026-08-...",None,Duplikat identik; satu salinan dipertahankan
5,penjualan_sintetis,PJ-00221,baris,"{""id_transaksi"":""PJ-00221"",""tanggal"":""2026-08-...",None,Duplikat identik; satu salinan dipertahankan
6,penjualan_sintetis,PJ-00221,baris,"{""id_transaksi"":""PJ-00221"",""tanggal"":""2026-08-...",None,Duplikat identik; satu salinan dipertahankan
7,penjualan_sintetis,PJ-00221,baris,"{""id_transaksi"":""PJ-00221"",""tanggal"":""2026-08-...",None,Duplikat identik; satu salinan dipertahankan
8,penjualan_sintetis,PJ-00221,baris,"{""id_transaksi"":""PJ-00221"",""tanggal"":""2026-08-...",None,Duplikat identik; satu salinan dipertahankan
9,penjualan_sintetis,PJ-00288,baris,"{""id_transaksi"":""PJ-00288"",""tanggal"":""2026-08-...",None,Duplikat identik; satu salinan dipertahankan


Pemeriksaan awal lulus. Omzet sintetis kandidat: 26490000
Pengeluaran sintetis kandidat: 10622000


## 8. Data Labeling

Pelabelan dilakukan untuk membedakan sumber data primer
yang diperoleh dari UMKM dengan data sintetis yang dibuat
untuk kebutuhan simulasi, analisis, dan pengujian aplikasi.

Label sumber data digunakan untuk menjaga transparansi
dan keterlacakan dataset.

In [23]:
# Labeling sumber data sintetis
jual_clean['sumber_data'] = 'sintetis'
belanja_clean['sumber_data'] = 'sintetis'

# Labeling sumber data primer
primer_jual['sumber_data'] = 'primer'
primer_belanja['sumber_data'] = 'primer'

# Pemeriksaan hasil labeling
display(jual_clean['sumber_data'].value_counts())
display(belanja_clean['sumber_data'].value_counts())
display(primer_jual['sumber_data'].value_counts())
display(primer_belanja['sumber_data'].value_counts())

sumber_data
sintetis    2573
Name: count, dtype: int64

sumber_data
sintetis    457
Name: count, dtype: int64

sumber_data
primer    11
Name: count, dtype: int64

sumber_data
primer    13
Name: count, dtype: int64

## 8. Ekspor hasil cleaning
Simpan kandidat ke `datasets/interim/`, sesuai kontrak notebook 03. Berkas primer review dan karantina disimpan terpisah. `datasets/processed/` tidak ditulis sampai validasi independen dan review manual selesai.

In [24]:
jual_clean.to_csv(INTERIM/'penjualan_sintetis_candidate.csv',index=False)
belanja_clean.to_csv(INTERIM/'pengeluaran_sintetis_candidate.csv',index=False)
reject_log.to_csv(INTERIM/'reject_log.csv',index=False)
change_log.to_csv(INTERIM/'change_log.csv',index=False)
primer_jual.to_csv(INTERIM/'penjualan_primer_review.csv',index=False)
primer_belanja.to_csv(INTERIM/'pengeluaran_primer_review.csv',index=False)
for dataset, source, key, filename in [
    ('penjualan_sintetis',raw_j,'id_transaksi','penjualan_sintetis_karantina.csv'),
    ('pengeluaran_sintetis',raw_b,'no','pengeluaran_sintetis_karantina.csv')]:
    source[source[key].isin(reject_log.loc[reject_log.dataset.eq(dataset),'id'])].to_csv(INTERIM/filename,index=False)
summary.to_csv(REPORTS/'cleaning_summary.csv',index=False)
print('Ekspor selesai. Lanjutkan 03_data_validation.ipynb untuk validasi final dan review log.')

Ekspor selesai. Lanjutkan 03_data_validation.ipynb untuk validasi final dan review log.
